# 106장 · 요청별 실행 추적과 저장 실패 진단

요청 ID를 따라가면 답변 생성과 저장이 각각 어디까지 끝났는지 추적할 수 있습니다. 단계가 일부만 성공한 경우도 같은 기록으로 인계합니다.

같은 요청의 UI·처리 함수·저장 기록을 요청 ID로 연결합니다. 마지막으로 성공한 단계와 재조회 결과에서 결함을 인계할 위치를 찾습니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 팀별 변형을 시도한 뒤 해당 셀 아래에서 예시를 펼쳐 비교할 수 있습니다.


## 같은 요청 ID로 처리 내역 재조회하기

105장의 저장 미확인 안내에서 사용한 요청 ID를 같은 소유자와 세션으로 찾습니다. 여기서는 임시 SQLite에 한 행만 넣어 조회가 성공하는 경우와 값이 없는 경우를 구분합니다.

이 프로젝트의 노트북은 한 폴더에서 열어 주세요. 첫 셀은 `team-project/final/chapter105.json`을 먼저 찾습니다. 팀 파일이 없으면 [이 장의 시작 예시](chapter106-start.json)를 노트북 옆에 내려받으세요. 출력 첫 줄에서 어느 파일을 읽었는지 확인할 수 있습니다.

In [ ]:
from pathlib import Path
import json
import re

here = Path.cwd()
project_root = here.parent if re.fullmatch(r"chapter[0-9]+", here.name) else here
team_file = project_root / "team-project/final/chapter105.json"
sample_file = here / "chapter106-start.json"
input_file = team_file if team_file.is_file() else sample_file
if not input_file.is_file():
    raise FileNotFoundError("이 장의 시작 예시를 내려받아 노트북 옆에 두세요.")
incoming = json.loads(input_file.read_text(encoding="utf-8"))
print("팀 기록" if input_file == team_file else "교재 예시", input_file.name)
print(incoming['storage'])

교재 예시 chapter106-start.json
미확인


두 번째 줄은 읽은 파일의 `storage` 값입니다.

## 요청 ID로 처리 기록 연결

전 구간 기록 예시　소유자 alice / 세션 eval-v1:Q01 / 요청 eval-v1:Q01 / 질문 장비 A 점검 / 처리 완료 completed / 답 answered와 RULE-A / 저장 committed / 새 저장소 객체로 다시 읽은 현재 응답 일치. 두 ID 문자열이 같은 것은 이 검사 자료에서만 쓴 표기입니다. 제품에서는 한 세션에서 서로 다른 요청을 여러 개 보낼 수 있습니다.


### 요청 ID로 제공 기록을 읽고 새 연결에서 대조하기

위쪽 예제는 미리 채운 기록을 읽습니다. 아래쪽 SQLite 실습에서는 연결을 새로 열고 같은 소유자·세션·요청을 사용해 Q01은 한 행, 저장하지 않은 요청은 행이 없는 것을 직접 확인합니다.

1. 요청 찾기 · request_id = eval-v1:Q01
2. 실행 연결 · request_id == run_id = True
3. 저장 기록 · storage_status = committed
4. 다시 읽기 · fresh_store_current_equal = True


### 셀 01 · 요청 ID로 처리 이벤트 대조

Q01 실행 기록의 대응 필드들입니다. 처리 작업의 run_id를 현재 request_id와 대조해 봅니다.

fresh_store_current_equal은 제공된 검사 기록의 판정입니다. 팀이 재현할 때는 저장 전 응답과 새 조회 응답에서 같은 턴의 답 문장과 출처 ID를 각각 꺼내 비교하고 그 결과를 적으세요.


In [1]:
trace = {'session_id': 'eval-v1:Q01',
 'request_id': 'eval-v1:Q01',
 'run_id': 'eval-v1:Q01',
 'storage_status': 'committed',
 'fresh_store_current_equal': True}
print(trace["request_id"] == trace["run_id"])
print(trace["storage_status"], trace["fresh_store_current_equal"])

True
committed True


첫째 줄은 제공한 기록에서 두 ID의 문자열이 같다는 뜻입니다. 둘째 줄은 기록에 미리 적힌 저장·재조회 결과를 읽은 값입니다. 이 셀 자체에는 파일이나 DB를 읽는 코드가 없습니다. 뒤의 SQLite 실습에서는 첫 연결을 닫고 다시 열어 Q01의 행을 직접 찾습니다.

여러 요청의 로그가 섞이면 연달아 찍힌 두 줄이 같은 작업인지 알 수 없습니다. 동일한 요청 ID를 기준으로 응답과 저장 기록을 연결해야 마지막으로 성공한 단계를 특정할 수 있습니다. 저장 실패가 남으면 인계 대상도 답변 생성 담당과 달라집니다.


#### 요청 ID로 제공 기록을 읽고 새 연결에서 대조하기

```mermaid
flowchart TD
  key["소유자 · 세션 · 요청"] --> readlog["제공된 추적 기록 읽기"]
  key --> db["새 SQLite 연결에서 조회"]
  db --> found{"해당 행이 있는가?"}
  found -->|있음| row["RULE-A"]
  found -->|없음| missing["None"]
```

위쪽 예제는 미리 채운 기록을 읽습니다. 아래쪽 SQLite 실습에서는 연결을 새로 열고 같은 소유자·세션·요청을 사용해 Q01은 한 행, 저장하지 않은 요청은 행이 없는 것을 직접 확인합니다.

## 부분 성공과 저장 실패 인계

저장 실패 예시　요청 save_failure / 계산은 완료 / storage_status failed / 오류 storage_failure / 새로 읽은 답의 일치 여부 미관측. 화면은 사용자에게 계산한 답이 아직 저장되지 않았다고 알려야 합니다. 요청 ID를 받은 운영 담당자는 같은 시도의 저장 오류부터 확인할 수 있습니다.


### 팀 참고 예시 · 실패 단계 인계


In [2]:
incident = {'request_id': 'save_failure',
 'storage_status': 'failed',
 'error': {'code': 'storage_failure',
           'stage': 'session_commit',
           'error_type': 'OSError'}}
print(incident["request_id"], incident["error"]["stage"], incident["error"]["code"])

save_failure session_commit storage_failure


save_failure, session_commit, storage_failure로 이어집니다. 원문 질문이나 민감한 본문 전체를 복사하지 않아도 해당 저장 시도를 좁혀 갈 수 있습니다. 두 True는 제공된 trace를 만든 실행에서 기록한 값입니다. 팀 DB에서도 확인했다면 새 연결에서 같은 ID로 읽은 값과 조회 시각을 별도로 남기세요.


#### 사용자 화면과 저장 로그를 함께 읽기

<mark>같은 요청 ID의 이벤트를 이어야 한 요청에서 어디까지 성공했는지 알 수 있습니다.</mark> 화면에 답이 표시됐더라도 session_commit 단계가 실패할 수 있습니다. 마지막 성공과 첫 실패를 기록해 답변 생성과 저장 중 어느 담당자가 확인할지 정합니다.

선수 개념 · 102장 POST 요청·응답 계약과 저장 상태

## 저장소를 닫았다 다시 열어 요청 기록 찾기

화면에서 답을 봤다는 기록과 저장된 대화를 다시 읽었다는 기록은 다릅니다. Q01의 결과를 새 파일에 넣고, 첫 연결을 닫은 다음 같은 사용자의 같은 요청으로 다시 조회합니다.

### 셀 04 · 요청 식별자 세 개를 사용해 재조회하기

이 셀에서는 임시 SQLite 파일로 저장 경계를 검증합니다. 기존 서비스의 JSON 저장소나 원격 API를 실행한 기록은 아닙니다.

In [3]:
import sqlite3
from pathlib import Path
from tempfile import TemporaryDirectory
from contextlib import closing

with TemporaryDirectory() as folder:
    file = Path(folder) / "requests.sqlite"
    with closing(sqlite3.connect(file)) as writer:
        writer.execute("CREATE TABLE replies(owner, session, request, doc, PRIMARY KEY(owner,session,request))")
        key = ("alice", trace["session_id"], trace["request_id"])
        writer.execute("INSERT INTO replies VALUES(?, ?, ?, ?)", (*key, "RULE-A"))
        writer.commit()
    with closing(sqlite3.connect(file)) as reader:
        statement = "SELECT doc FROM replies WHERE owner=? AND session=? AND request=?"
        print(reader.execute(statement, key).fetchone())
        print(reader.execute(statement, (*key[:2], "save_failure")).fetchone())

('RULE-A',)
None


새 연결로도 Q01의 행을 찾고, 기록하지 않은 save_failure는 None이 나옵니다. 조회 결과 없음만으로는 왜 저장되지 않았는지 알 수 없습니다. 그 원인은 incident의 단계 기록과 대조합니다.

### 같은 세션에서 두 요청 구분하기

팀 서비스에서는 세션 ID 하나에 질문이 여러 개 들어옵니다. 두 번째 요청의 ID를 별도로 만들어 같은 세션에 쓰고, 첫 요청을 다시 읽었을 때 처음 근거가 남는지 확인합니다. 내 화면에서 직전 답 한 개만 재사용해도 되는 상황과 구분하세요.

<mark>재현하려는 요청에는 소유자, 세션, 요청 식별자가 모두 필요합니다.</mark> 예시 Q01에서 session_id와 request_id가 같은 문자열이어도 실제 서비스에서 둘을 같은 키로 써야 한다는 뜻은 아닙니다.

107장 관찰표에는 어느 입력에서 나온 어떤 카드였는지 요청 ID와 함께 남깁니다.

## 팀 사례에 적용하기 · 같은 요청 ID로 처리 내역 재조회하기

위 예제에 첫 셀에서 읽어 둔 팀 기록을 넣어 봅니다.

In [ ]:
with sqlite3.connect(":memory:") as db:
    db.execute("CREATE TABLE events(owner, session, request_id)")
    key = ("팀A", "main", incoming["reply"]["request_id"])
    db.execute("INSERT INTO events VALUES(?, ?, ?)", key)
    sql = "SELECT count(*) FROM events WHERE owner=? AND session=? AND request_id=?"
    saved = db.execute(sql, key).fetchone()[0]
    other = db.execute(sql, ("팀B", *key[1:])).fetchone()[0]
new = {**incoming, "lookup": {"request": key[2], "own_rows": saved, "other_rows": other,
                             "storage": "수업 임시 SQLite"}}
print(saved, other)

1 0


실습 DB에서 찾은 한 행으로 수업 서버의 저장 여부까지 알 수는 없습니다. 운영 담당에게 넘길 때는 추적할 ID와 확인한 저장소를 함께 남깁니다.

In [ ]:
output_file = project_root / "team-project/final/chapter106.json"
output_file.parent.mkdir(parents=True, exist_ok=True)
with output_file.open("a+", encoding="utf-8") as handle:
    handle.seek(0)
    previous_text = handle.read()
    if previous_text and json.loads(previous_text) != new:
        raise FileExistsError("같은 장의 다른 결과가 있습니다. 기존 JSON을 다른 이름으로 보관한 뒤 다시 실행하세요.")
    if not previous_text:
        json.dump(new, handle, ensure_ascii=False, indent=2)
        handle.write("\n")
print(output_file.relative_to(project_root).as_posix())

team-project/final/chapter106.json


107장 첫 셀이 이 파일을 읽습니다. 입력을 바꿔 다시 실행할 때는 기존 결과 파일의 이름을 바꿔 보관하세요.

## 돌아보기

베타 점검에는 정상 요청 한 건과 저장 실패 한 건을 가져가세요. 각각 어떤 소유자·세션·요청 ID의 기록인지 적어 둡니다. 이 예시는 교재의 로컬 Python·JSON 기록입니다. 실제 로그인·DB·HTTP·사용자 클릭은 별도 시험이 필요합니다.
